# UMBC Soccer — Opponent Strength & Contextual xG Modeling (2023–2025)

**Author:** Jervon Drakes  
**Purpose:** Multi-season contextual modeling of expected goals (xG) using opponent strength and conference context.

This notebook:
- Loads match-level team data from 2023–2025
- Engineers context features (home, shot accuracy, conference indicator)
- Computes **opponent strength** without data leakage (pre-match, based only on prior UMBC vs opponent results with shrinkage)
- Fits a **contextual Poisson regression** to estimate match-level expected goals (**xG_ctx**)
- Summarizes differences between conference and non-conference performance



## 1. Motivation & Modeling Philosophy

A baseline xG model based only on shot counts (or shots on goal) can miss key context:
- Some opponents are consistently harder to score against
- Conference matches are often more tactically disciplined and familiar
- Home/away environment can shift attacking output

This notebook extends xG modeling by explicitly adding:
- **Opponent strength (pre-match, no leakage)**
- **Conference vs non-conference indicator**
- **Conference × opponent strength interaction**

Model form:

$
\log(E[\text{goals\_for}]) =
\beta_0 + \beta_1 \cdot \text{SOG} + \beta_2 \cdot \text{Home} + \beta_3 \cdot \text{Accuracy}
+ \beta_4 \cdot \text{OppStrength} + \beta_5 \cdot \text{Conference} + \beta_6 \cdot (\text{Conference} \times \text{OppStrength})
$

The goal is a stable, interpretable estimate of **expected goals per match** that adjusts for difficulty and environment.



## 2. Data Sources & Integration

I load match-level data from 2023, 2024, and 2025 and combine them into one dataset.
Combining seasons increases sample size and improves stability of both opponent strength estimates and model coefficients.



In [25]:
import pandas as pd
import numpy as np


# 0) File inputs (edit these)

CSV_FILES = [
    "team_match_2023.csv",
    "team_match_2024.csv",
    "team_match_2025.csv",
]


# 1) Load + standardize

dfs = []
for f in CSV_FILES:
    df = pd.read_csv(f)

    # Ensure season exists (if missing, infer from filename)
    if "season" not in df.columns:
        import re
        m = re.search(r"(20\d{2})", f)
        if m:
            df["season"] = int(m.group(1))

    dfs.append(df)

matches = pd.concat(dfs, ignore_index=True)

# Standardize column names
matches.columns = [c.strip() for c in matches.columns]

# Parse date (handles "8/21/2025" style)
matches["date"] = pd.to_datetime(matches["date"], errors="coerce")

# Sort chronologically
matches = matches.sort_values(["date", "season"]).reset_index(drop=True)

matches.head()



,season,date,home_away,opponent,result,goals_for,goals_against,attendance,shots,shots_on_goal,yellow,red
0,2023.0,2023-08-24,Home,George Mason,W,4,3,824,14,7,4,0
1,NaN,2023-08-28,Home,Old Dominion,T,1,1,2891,17,7,3,0
2,NaN,2023-09-01,Home,Liberty,W,3,0,972,9,5,0,0
3,NaN,2023-09-03,Home,Howard,W,2,0,1021,18,12,0,0
4,NaN,2023-09-12,Away,Hofstra,L,0,2,303,6,3,2,1


In [26]:
matches[["season", "date", "opponent", "home_away", "goals_for", "goals_against", "shots", "shots_on_goal"]].head(10)


,season,date,opponent,home_away,goals_for,goals_against,shots,shots_on_goal
0,2023.0,2023-08-24,George Mason,Home,4,3,14,7
1,NaN,2023-08-28,Old Dominion,Home,1,1,17,7
2,NaN,2023-09-01,Liberty,Home,3,0,9,5
3,NaN,2023-09-03,Howard,Home,2,0,18,12
4,NaN,2023-09-12,Hofstra,Away,0,2,6,3
5,NaN,2023-09-16,Cornell,Home,2,1,6,4
6,NaN,2023-09-19,American,Home,1,3,18,4
7,NaN,2023-09-23,Ualbany,Away,2,1,9,3
8,NaN,2023-09-30,Bryant,Home,0,0,6,1
9,NaN,2023-10-03,Navy,Home,4,4,21,14


## 3. Feature Engineering & Context Variables

I engineer the core context variables used in the model:

- **home:** 1 for home matches, 0 otherwise  
- **shot_accuracy:** shots_on_goal / shots (safe divide), used for chance quality  
- **opponent cleaning:** normalize opponent names to reduce join/label issues  

These variables support more interpretable and stable contextual xG estimates.



In [27]:

# 2) Feature engineering

# Home flag
matches["home"] = (matches["home_away"].astype(str).str.strip().str.lower() == "home").astype(int)

# Shot accuracy (safe divide)
matches["shot_accuracy"] = np.where(
    matches["shots"].fillna(0) > 0,
    matches["shots_on_goal"] / matches["shots"],
    0.0
).astype(float)

# Clean opponent names
matches["opponent"] = matches["opponent"].astype(str).str.strip()

matches[["date","season","opponent","home_away","home","shots","shots_on_goal","shot_accuracy"]].head(10)



,date,season,opponent,home_away,home,shots,shots_on_goal,shot_accuracy
0,2023-08-24,2023.0,George Mason,Home,1,14,7,0.500000
1,2023-08-28,NaN,Old Dominion,Home,1,17,7,0.411765
2,2023-09-01,NaN,Liberty,Home,1,9,5,0.555556
3,2023-09-03,NaN,Howard,Home,1,18,12,0.666667
4,2023-09-12,NaN,Hofstra,Away,0,6,3,0.500000
5,2023-09-16,NaN,Cornell,Home,1,6,4,0.666667
6,2023-09-19,NaN,American,Home,1,18,4,0.222222
7,2023-09-23,NaN,Ualbany,Away,0,9,3,0.333333
8,2023-09-30,NaN,Bryant,Home,1,6,1,0.166667
9,2023-10-03,NaN,Navy,Home,1,21,14,0.666667


## 4. Conference vs Non-Conference Context

Conference competition is often tactically distinct due to:
- repeated matchups and scouting familiarity
- more conservative game states
- increased defensive organization

I label matches as conference vs non-conference using an opponent list
(America East opponents), then create a binary indicator **is_conference**.



In [28]:

# 3) Conference vs non-conference (America East)

CONFERENCE_OPPONENTS = {
    "Vermont",
    "Bryant",
    "Binghamton",
    "NJIT",
    "Umass Lowell",
    "New Hampshire",
    "Ualbany",
    "Maine",
}

matches["is_conference"] = matches["opponent"].isin(CONFERENCE_OPPONENTS).astype(int)

matches[["date","opponent","is_conference"]].head(15)



,date,opponent,is_conference
0,2023-08-24,George Mason,0
1,2023-08-28,Old Dominion,0
2,2023-09-01,Liberty,0
3,2023-09-03,Howard,0
4,2023-09-12,Hofstra,0
5,2023-09-16,Cornell,0
6,2023-09-19,American,0
7,2023-09-23,Ualbany,1
8,2023-09-30,Bryant,1
9,2023-10-03,Navy,0


## 5. Estimating Opponent Strength (No Leakage + Shrinkage)

I estimate **opponent strength** as a pre-match difficulty score using only UMBC’s
prior results against that opponent (no data leakage).

Approach:
- For each match, look only at **previous** matches vs the same opponent
- Compute prior goal difference: GD = goals_for − goals_against
- Tough opponents produce worse historical GD for UMBC
- Apply shrinkage toward 0 so small samples don’t dominate

Shrinkage formula:

$ opp\_strength = - \frac{\sum GD_{prev}}{n_{prev} + m} $



Where:
- \(m\) is the shrinkage strength (typical range 1–5)

I also create:
- **conf_x_opp = is_conference × opp_strength**
to capture whether opponent strength behaves differently in conference play.


In [29]:

# 4) Opponent strength (pre-match, shrinkage; NO leakage)

matches = matches.sort_values("date").reset_index(drop=True)

m_prior = 3.0  # shrinkage strength (tune 1–5)
strength = []

for i, row in matches.iterrows():
    prev = matches[(matches["opponent"] == row["opponent"]) & (matches["date"] < row["date"])].copy()
    if len(prev) == 0:
        strength.append(0.0)
    else:
        gd_prev = (prev["goals_for"] - prev["goals_against"]).astype(float)
        s = -(gd_prev.sum() + m_prior * 0.0) / (len(gd_prev) + m_prior)
        strength.append(float(s))

matches["opp_strength"] = strength
matches["conf_x_opp"] = matches["is_conference"] * matches["opp_strength"]

matches[["date","opponent","goals_for","goals_against","opp_strength","is_conference","conf_x_opp"]].head(20)



,date,opponent,goals_for,goals_against,opp_strength,is_conference,conf_x_opp
0,2023-08-24,George Mason,4,3,0.0,0,0.0
1,2023-08-28,Old Dominion,1,1,0.0,0,0.0
2,2023-09-01,Liberty,3,0,0.0,0,0.0
3,2023-09-03,Howard,2,0,0.0,0,0.0
4,2023-09-12,Hofstra,0,2,0.0,0,0.0
5,2023-09-16,Cornell,2,1,0.0,0,0.0
6,2023-09-19,American,1,3,0.0,0,0.0
7,2023-09-23,Ualbany,2,1,0.0,1,0.0
8,2023-09-30,Bryant,0,0,0.0,1,0.0
9,2023-10-03,Navy,4,4,0.0,0,0.0


## 6. Contextual Poisson xG Model (Training)

Goals are count data, so we fit a Poisson regression model to estimate expected goals per match.

Predictors:
- shots_on_goal (volume)
- home (home advantage)
- shot_accuracy (quality)
- opp_strength (difficulty)
- is_conference (competition environment)
- conf_x_opp (interaction: opponent strength effect in conference play)

Output:
- **xG_ctx**: context-adjusted expected goals for each match


In [30]:
from sklearn.linear_model import PoissonRegressor

X_attack = matches[[
    "shots_on_goal",
    "home",
    "shot_accuracy",
    "opp_strength",
    "is_conference",
    "conf_x_opp",
]].copy()

y_attack = matches["goals_for"].astype(float)

poisson_attack = PoissonRegressor(alpha=0.0, max_iter=10000)
poisson_attack.fit(X_attack, y_attack)

matches["xG_ctx"] = poisson_attack.predict(X_attack)

coef_map = dict(zip(X_attack.columns, poisson_attack.coef_))
print("Intercept:", poisson_attack.intercept_)
print("Coefficients:", coef_map)


Intercept: -0.7639493770140006
Coefficients: {'shots_on_goal': 0.1248628848117557, 'home': 0.1087044054866404, 'shot_accuracy': 0.5844200414679641, 'opp_strength': 0.001464732273064716, 'is_conference': -0.07471550053702622, 'conf_x_opp': -0.8320436389699646}


## 7. Coefficient Interpretation (Multipliers)

Poisson regression coefficients operate in log space.
To interpret effects in intuitive soccer terms, we exponentiate coefficients:

$
\text{multiplier} = e^{\beta}
$

Example:
- multiplier = 1.10 → ~10% increase in expected goals per unit increase
- multiplier = 0.90 → ~10% decrease in expected goals per unit increase
- 
I compute multipliers for each predictor to interpret tactical and contextual effects.


In [31]:
mult = {k: float(np.exp(v)) for k, v in coef_map.items()}
print("Multipliers (exp(coef)):")
print(mult)


Multipliers (exp(coef)):
{'shots_on_goal': 1.1329930918548097, 'home': 1.1148327631836592, 'shot_accuracy': 1.7939502671188141, 'opp_strength': 1.0014658055173218, 'is_conference': 0.9280074664015048, 'conf_x_opp': 0.4351590689584609}


## 8. Conference vs Non-Conference Comparison

I summarize average attacking output by match type (conference vs non-conference) using:
- goals_for
- xG_ctx
- shots_on_goal
- shot_accuracy

This helps quantify whether conference play suppresses attacking output and whether that suppression is reflected in expected goals.


In [32]:
summary = matches.groupby("is_conference")[["goals_for", "xG_ctx", "shots_on_goal", "shot_accuracy"]].mean()
summary.index = ["Non-Conference", "Conference"]
summary


,goals_for,xG_ctx,shots_on_goal,shot_accuracy
Non-Conference,1.571429,1.571412,6.107143,0.451506
Conference,1.250000,1.250045,5.000000,0.419656


In [ ]:
### Conference vs Non-Conference Goal Differential

To quantify the impact of conference competition on match difficulty, we compute UMBC’s **average goal differential (GD)** separately for conference and non-conference matches.

Goal differential is defined as:

`goal_diff = goals_for − goals_against`

This provides a holistic measure of match dominance, capturing both offensive output and defensive resistance.




In [33]:
# Compute goal differential
matches["goal_diff"] = matches["goals_for"] - matches["goals_against"]

# Average GD by conference vs non-conference
gd_summary = matches.groupby("is_conference")["goal_diff"].agg([
    "count", "mean", "median", "std"
])

gd_summary.index = ["Non-Conference", "Conference"]
gd_summary


,count,mean,median,std
Non-Conference,28,0.178571,0.0,1.866936
Conference,24,0.208333,0.0,1.531670


### Conference vs Non-Conference Goal Differential

We compute average **goal differential (GD)** to assess whether conference play leads to worse match outcomes overall.

**Results:**

- Non-Conference Mean GD: **+0.18**
- Conference Mean GD: **+0.21**

This indicates that **UMBC performs similarly in overall match outcomes across both match types**.

However, match variability differs:

- Non-conference matches show **higher variance**, indicating more volatile outcomes.
- Conference matches exhibit **lower variance**, suggesting tighter, more tactically controlled games.

**Interpretation:**

Although conference matches suppress attacking output (lower shots, lower xG, lower accuracy), UMBC appears to **compensate defensively**, maintaining stable match outcomes.

This suggests a **tactical adaptation effect**:

> In conference play, UMBC sacrifices attacking volume in exchange for defensive stability and match control.

This dynamic explains why the model identifies **conference play as suppressing expected goals**, while goal differential remains largely unchanged.


## 9. Key Findings

### 9.1 Shot Volume and Shot Quality Drive Attacking Output

The model confirms that **chance creation is the dominant driver of goal scoring**:

- **Shots on goal multiplier ≈ 1.13**
  - Each additional shot on goal increases expected goals by **~13%**.
  - This confirms that **shot volume is a primary driver of xG.**

- **Shot accuracy multiplier ≈ 1.79**
  - A 10 percentage point increase in shot accuracy raises expected goals by roughly **8–9%**.
  - This shows that **chance quality (shot selection and placement)** is a major determinant of scoring.

**Interpretation:**  
UMBC’s attacking output is strongly driven by *how often* and *how cleanly* shots are generated, emphasizing the importance of chance creation patterns, buildup quality, and shot selection.

---

### 9.2 Home Advantage Effects

- **Home multiplier ≈ 1.11**

Playing at home increases expected goals by approximately **11–12%**.

**Interpretation:**  
Home-field advantage appears meaningful, likely reflecting:
- familiarity with field and surroundings,
- crowd support,
- travel fatigue effects on opponents.

This suggests that **home matches systematically favor attacking production.**

---

### 9.3 Conference vs Non-Conference Context

- **Conference indicator multiplier ≈ 0.93**

Conference matches reduce expected goals by roughly **7%**, holding other variables constant.

This is supported by raw averages:

| Match Type | Goals | xG_ctx | Shots on Goal | Shot Accuracy |
|-------------|--------|----------|----------------|----------------|
| Non-Conference | 1.57 | 1.57 | 6.11 | 45.2% |
| Conference | 1.25 | 1.25 | 5.00 | 42.0% |

**Interpretation:**
- Conference matches are:
  - more tactically disciplined,
  - more defensively structured,
  - less open.
- UMBC produces **fewer and lower-quality chances** in conference play, leading to lower expected goals.

This confirms that **conference competition systematically suppresses attacking output.**

---

### 9.4 Opponent Strength Effects

- **Opponent strength multiplier ≈ 1.001**

On its own, opponent strength shows a **small average effect**. This is expected because:

- The strength metric is built using only UMBC historical matchups.
- Sample sizes per opponent remain limited.
- Many opponents are faced only once or twice.

However, the interaction term:

- **Conference * Opponent Strength multiplier ≈ 0.44**

This implies that **strong opponents in conference play suppress UMBC’s expected goals by over 50% more** than strong non-conference opponents.

**Interpretation:**
- Strong conference opponents are:
  - more familiar tactically,
  - better prepared defensively,
  - more capable of limiting shot quality and volume.
- This confirms that **match difficulty increases sharply in conference play.**

---

### 9.5 Tactical Summary

Overall, the model suggests:

1. **Chance creation is the primary driver of goal scoring.**
2. **Home advantage meaningfully boosts attacking output.**
3. **Conference play suppresses attacking production.**
4. **Strong conference opponents are especially effective at limiting chances.**

This implies that:
- Tactical focus should prioritize **chance creation mechanisms**, especially in:
  - away matches,
  - conference fixtures,
  - games against top conference opponents.

---

### 9.6 Strategic Implications for Coaching & Analysis

The model suggests several practical insights:

- **Non-conference matches:**
  - Emphasize aggressive attacking schemes.
  - Higher openness allows better exploitation of shot volume and accuracy.

- **Conference matches:**
  - Tactical emphasis should shift toward:
    - set-piece efficiency,
    - transition attacks,
    - high-value shot selection.
  - Because **overall chance volume is lower**, finishing efficiency and tactical discipline become more critical.

- **Opponent-specific scouting:**
  - Against top conference defenses, **quality > quantity** in shooting decisions.

---

### 9.7 Modeling Impact

By incorporating opponent strength and conference context, this model:
- Produces **more realistic expected goals estimates (xG_ctx)**,
- Allows fairer comparison across matches of varying difficulty,
- Supports deeper tactical interpretation than simple shot-based xG models.

This contextual xG framework forms the foundation for:
- performance diagnostics,
- tactical evaluation,
- match simulation,
- and future defensive modeling.


## 10. Match-Level Diagnostics & Tactical Case Studies

While aggregate model coefficients quantify general attacking trends, **match-level diagnostics allow us to interpret tactical and execution-based factors that influence individual game outcomes.**

We analyze selected matches using:
- Contextual expected goals (**xG_ctx**)
- Actual goals scored (**goals_for**)
- Finishing differential (**goals_for − xG_ctx**)
- Opponent strength
- Conference vs non-conference context

This approach helps identify **where performance exceeded or fell short of statistical expectations**, and whether those deviations reflect tactical execution, finishing variance, or opponent difficulty.

---

### 10.1 High Overperformance: UMBC vs Navy (2023-10-03, Home)

**Match Summary:**
- Opponent: Navy (Non-Conference)
- Result: 4–4 Draw  
- xG_ctx: 4.40  
- Goals: 4  
- Finishing Diff: −0.40  

**Interpretation:**

This match represents a **high-volume attacking performance**. UMBC generated:
- 21 shots  
- 14 shots on goal  
- High shot accuracy (67%)

Despite scoring four goals, **finishing slightly underperformed expected goals**, indicating:

> The attack generated **elite chance volume**, but finishing efficiency was slightly below statistical expectation.

This suggests:
- Tactical system successfully created chances  
- Finishing variance prevented a decisive victory  

---

### 10.2 Extreme Finishing Overperformance: UMBC vs Binghamton (2025-10-31, Home)

**Match Summary:**
- Opponent: Binghamton (Conference)
- Result: 4–0 Win  
- xG_ctx ≈ 1.2  
- Goals: 4  
- Finishing Diff: +2.8  

**Interpretation:**

This match demonstrates **extreme finishing overperformance**. UMBC:
- Converted modest chance volume into **four goals**
- Outperformed expected scoring by **nearly three goals**

This likely reflects:
- Exceptional shot placement  
- Defensive breakdowns by opponent  
- Potential goalkeeper underperformance  

This type of match illustrates:

> **Finishing variance dominates single-match outcomes**, even when tactical indicators remain moderate.

---

### 10.3 Finishing Underperformance: UMBC vs Old Dominion (2025-09-12, Home)

**Match Summary:**
- Opponent: Old Dominion (Non-Conference)
- Result: 0–1 Loss  
- xG_ctx ≈ 1.1  
- Goals: 0  
- Finishing Diff: −1.1  

**Interpretation:**

UMBC generated sufficient chance quality to score approximately **one goal**, yet failed to convert.

This suggests:
- Finishing inefficiency  
- Strong opposing goalkeeping  
- Shot outcome variance  

Such matches reinforce:

> Tactical chance creation does not guarantee goal output in small samples.

---

### 10.4 Tactical Implications from Case Studies

These match diagnostics reveal:

| Observation | Tactical Meaning |
|--------------|------------------|
| High xG, low goals | Finishing variance dominates |
| Low xG, high goals | Clinical finishing, defensive breakdowns |
| High SOG + accuracy | Tactical structure is effective |
| Conference suppression | Defensive compactness reduces chance quality |

Overall:

> **Chance creation is structurally consistent, while finishing fluctuates stochastically.**

This supports the statistical finding that:
- Shot generation + accuracy explain most systematic scoring variation  
- Finishing remains largely **random at the match level**

---

### 10.5 Coaching & Tactical Applications

This analysis enables coaching staff to:

- Separate **tactical inefficiency** from **finishing randomness**
- Avoid overreacting to low-scoring matches with strong xG profiles  
- Identify matches requiring:
  - finishing drills  
  - movement pattern refinement  
  - shot selection optimization  

---

**Key Takeaway:**

> UMBC’s attack is **structurally sound in chance creation**, with match outcomes primarily driven by finishing variance and opponent context.
